# 00 · Phân công và cách chạy

Tách từ `(thanhbangcao)FULL_PIPELINE_AIC_2026.ipynb` (3 cell khổng lồ) thành 8 notebook
chạy độc lập. **Code lấy nguyên văn**, chỉ thay khối đường dẫn và thêm ô chọn phạm vi.


| Notebook | Hộp trong sơ đồ offline | Ai chạy |
|---|---|---|
| `01_cat_keyframe_cach1` | `Videos → TransNetV2 → Scenes → Keyframes` — cố định 4 ảnh, **để đối chứng** | ai muốn so |
| `02_cat_keyframe_cach2` | như trên, **lấy mẫu theo tốc độ — cách đang dùng** | mọi người |
| `03_index_beit3_clip`   | `CLIP/BEiT3 Image Encoder → embedding → Faiss` | mọi người |
| `04_test_ocr`           | `OCR (PaddleOCR ⇒ VietOCR) → ocr_result` | Vanh |
| `05_test_caption`       | `Caption/Description (Vintern) → caption_result` | Phat |
| `06_test_tag`           | `Tag/Keyword (RAM++/Vintern) → keyword_result` | An |
| `07_test_object_color`  | hộp `...` còn để trống trên sơ đồ | Bang |
| `08_test_asr`           | `Videos → Audio → ASR (Whisper) → asr_result` | ai rảnh GPU |


> **Chạy trong một phiên Colab RIÊNG.** Thư viện giữa các notebook xung đột nhau
> (paddleocr/vietocr hạ cấp Pillow), nên đừng chạy hai notebook trong cùng một runtime.

**Sửa đúng một ô: `SECTION 02a`.** Đổi `NGUOI` thành tên mình, chọn `PHAM_VI`, đặt
`SO_VIDEO`. Mọi đường dẫn tự suy ra từ đó.

```
/content/drive/MyDrive/aic_26/
├── fps_map.json                     ← dùng chung, chép từ data_raw/ trong repo
├── transnetv2-pytorch-weights.pth   ← dùng chung, tự tải lần đầu
└── output_demo/
    ├── Vanh/v001/     ← mỗi người một nhánh, không giẫm chân nhau
    │   ├── progress.json
    │   ├── zips/      AIC2026_frames_001.zip …
    │   └── indexes/   beit3.index  clip.index  keyframe_metadata.json
    │                  keyframes_list.json  image_mapping.json  fps_map.json
    ├── Phat/v001/
    ├── An/v001/
    └── Bang/v001/
```


## Chuẩn bị một lần cho cả nhóm

Chép hai file lên `/content/drive/MyDrive/aic_26/`:

| File | Lấy ở đâu |
|---|---|
| `fps_map.json` | `data_raw/fps_map.json` trong repo — 873 video L21–L30 |
| `transnetv2-pytorch-weights.pth` | notebook tự tải lần đầu, để đó dùng chung |

Thiếu `fps_map.json` thì cách 2 dừng ngay với thông báo rõ ràng, không cắt sai.

## Thứ tự chạy

```
01 hoặc 02   cắt keyframe   →   03   dựng index   →   04…08   test model
```

Bốn notebook test (04–07) đã **có sẵn bước cắt và bước index bên trong**, dùng **cách 2**,
nên mỗi người chạy một notebook duy nhất là ra đủ chuỗi cho phần video của mình.
`08_test_asr` không cắt keyframe — ASR đọc thẳng file video.

## Cách cắt đang dùng: **cách 2**

Đây là cách đã chốt, nằm trong mọi notebook test.

```
n = 2 ảnh          cho tới 1,67 giây
sau đó             cứ 2 giây thêm 1 ảnh
kẹp                [2, 40]
lùi biên           3 frame mỗi đầu
KHÔNG lọc trùng    ở bước cắt
fps                lấy từ bảng BTC, không dò từ video
```

Quy ra frame @30fps: `8–50 → 2 ảnh · 51–110 → 3 · 111–170 → 4`
Quy ra frame @25fps: `8–42 → 2 ảnh · 43–92 → 3 · 93–142 → 4`

Mốc theo **giây** giống nhau ở mọi fps, nên video 25 và 30 fps có cùng mật độ ảnh
trên mỗi giây.

## Cách 1 giữ lại để đối chứng

`01_cat_keyframe_cach1` là bản y hệt notebook gốc, cố định 4 ảnh mỗi cảnh.

| | Cách 1 (`01`) | **Cách 2 (`02` và mọi notebook test)** |
|---|---|---|
| Số ảnh mỗi cảnh | **cố định 4** | **theo độ dài cảnh** |
| Cảnh 8 frame | 4 ảnh | 2 ảnh |
| Cảnh 4500 frame | 4 ảnh | 40 ảnh |
| Lọc trùng | có, CLIP ViT-B/32 trên ảnh 48×27 | **không** — paper §3.2.3 lọc sau khi trích đặc trưng |
| Nguồn fps | `ffprobe` dò từ video | **bảng của BTC** |
| Lùi biên | không | 3 frame mỗi đầu |
| Model phải nạp | thêm CLIP ViT-B/32 | bớt được một model |

Đo trên `L21_V015` (39117 frame, 21,7 phút): cách 2 cho 331 cảnh → **1200 ảnh**,
mật độ 1 ảnh/1,09 giây, cảnh dài nhất được 10 ảnh thay vì 4.

Muốn so hai cách thì chạy `01` và `02` trên **cùng một phần video**, nhớ đổi `v001`
thành `v002` cho lần thứ hai để khỏi ghi đè lên nhau.